In [4]:
import os
from langchain_openai import ChatOpenAI
from crewai_tools import YoutubeVideoSearchTool
from langchain_community.tools.tavily_search import TavilySearchResults
from crewai.tools import tool
from crewai import Crew, Task, Agent
from dotenv import load_dotenv

load_dotenv()

True

In [5]:
from langchain_community.tools import YouTubeSearchTool
youtube_search_tool = YouTubeSearchTool()
youtube_search_tool.run('테디노트')

"['https://www.youtube.com/watch?v=zybyszetEcE&pp=ygUM7YWM65SU64W47Yq4', 'https://www.youtube.com/watch?v=aMUopbBrAmA&pp=ygUM7YWM65SU64W47Yq4']"

In [3]:
llm = ChatOpenAI(model='gpt-4o-mini')

In [8]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import YoutubeLoader
from langchain_core.documents import Document
import ast

youtube_search_tool = YouTubeSearchTool()

@tool
def youtube_retriever(query:str) -> str:
    '''
    Retriever tool for the transcript of a Youtube video.
    query should be search_query with str format which can retriever appropriate youtube video.
    '''
    urls = youtube_search_tool.run(query)
    urls = ast.literal_eval(urls)

    docs = []
    for url in urls:
        loader = YoutubeLoader.from_youtube_url(
            url,
            add_video_info=True,
            language=['en', 'ko']
        )
        scripts = loader.load()
        script_content = scripts[0].page_content
        title=scripts[0].metadata['title']
        author=scripts[0].metadata['author']
        doc = Document(page_content=script_content, metadata={'source':url, 'title':title, 'author': author})
        docs.append(doc)

    text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
    texts = text_splitter.split_documents(docs)
    embeddings = OpenAIEmbeddings()
    db = FAISS.from_documents(texts, embeddings)
    retriever = db.as_retriever()
    retrieved_docs = retriever.invoke(query)

    video_results = []

    for doc in retrieved_docs:
        title = doc.metadata.get('title', 'No title available')
        author = doc.metadata.get('author', 'No author available')
        script_content = doc.page_content

        video_info = f"""
        Video Information:
        -------------------
        Title: {title}
        Author: {author}
        Transcript:
        {script_content}
        -------------------
        """
        video_results.append(video_info)

    all_video_results = '\n\n'.join(video_results)

    return all_video_results

In [9]:
web_search_tool = TavilySearchResults(k=3)

In [12]:
@tool
def web_search(query: str) -> str:
    '''
    Web search tool using Tavily API.
    query should be a search query string to find information on the web.
    '''
    tavily = TavilySearchResults(k=3)
    results = tavily.invoke(query)
    return str(results)

In [13]:
query = 'langgraph가 무엇인가요?'
video_analyzer = Agent(
    role='Video Analyzer',
    goal = f"""
    Analyze youtube videos about user's query: {query} and Analyze Youtube video transcripts and identify main topics,
    key points, and areas needing further research. This is crucial for answering user's query.
    """,
    backstory='Expert in Youtube video analysis with a keen eye for identifying core themes and knowledge gaps',
    verbose=True,
    max_iter=2,
    llm=llm,
    tools=[youtube_retriever]
)

researcher = Agent(
    role='Web Researcher',
    goal=f"Conduct web searches with query to find additional information on topics identified from the video to answer the user's query: {query}",
    backstory='Skilled internet researcher with a talent for finding reliable and relevant information quickly',
    verbose=True,
    llm=llm,
    max_iter=2,
    tools=[web_search]
)

rag_agent = Agent(
    role='RAG Agent',
    goal=f'''Answer user's query: {query} based on video content analysis and addition research
    if resources are not enough to answer the users' question, then you should command other agents for further research.
    ''',
    backstory='You are a helpful RAG Agent who should refer to the analysis of video analyzer and researcher.',
    llm=llm,
    verbose=True
)

In [21]:
task1 = Task(
    description=f"""Analyze the video transcript.
    identify main topics, key points for each topic, and
    list questions that need further research to answer user's {query}.""",
    agent=video_analyzer,
    expected_output="A detailed analysis of the video content including main topics, key points, and questions for answering user's query."
)

task2 = Task(
    description=f"""Research the user query: {query} identified from the video analysis and provide findings with sources.
    search query for web search tool should be string format.""",
    agent=researcher,
    expected_output="Comprehensive research findings for each identified question, including relevant information and sources(url needed)."
)

task3 = Task(
    description=f"""Answer the user's query: {query} with factful resources from video and web search result.
    you should consider user's language to give great answer.
    """,
    agent=rag_agent,
    expected_output=f"""A well-structured, engaging and concise answer to user's query: {query} based on the video content and additional research,
    including a title, main content, and references(including URLs)."""
)

In [22]:
crew = Crew(
    agents=[video_analyzer, researcher, rag_agent],
    tasks=[task1, task2, task3],
    verbose=True
)

In [ ]:
result = crew.kickoff()

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 8afd1c50-3507-4d4a-a02b-acec97437ed8                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Video Analyzer                                                                                          │
│                                                                                                                 │
│  Task: Analyze the video transcript.                                                                            │
│      identify main topics, key points for each topic, and                                                       │
│      list questions that need further research to answer user's langgraph가 무엇인가요?.                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

Output()

╭────────────────────────────────────────────────── Tool Error ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Usage Failed                                                                                              │
│  Name: youtube_retriever                                                                                        │
│  Error: HTTP Error 400: Bad Request                                                                             │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: RAG Agent                                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Langgraph란 무엇인가요?                                                                                    │
│                                                                                                                 │
│  Langgraph는 언어 처리 도구로, 다양한 언어적 데이터를 분석하고 시각화할 수 있는 그래프 및 네트워크 기반의       │
│  플랫폼입니다. 이를 통해 사용자는 구문, 의미론, 음성학 등 언어의 다양한 요소 간의 관계를 이해할 수 있습니다.    │
│  Langgraph는 LangChain 에코시스템의 중요한 부분으로, 복잡한 생성적 AI 에이전트 워크플로를 구축하고 배포하는 데  │
│  특화되어 있습니다.                                                                                             │
│                                                                                                                 │
│  #### 주요 기능 및 방법론                                                                                       │
│  1. **그래프 기반 아키텍처**: Langgraph는 노드가 다양한 작업이나 구성 요소(예: LLM과 상호작용, 데이터 조작      │
│  수행)를 나타내고, 엣지가 이들 간의 관계와 워크플로를 정의하는 그래프 모델을 사용합니다. 이 구조는 복잡한       │
│  작업과 모델 및 API 간의 상호작용을 효율적으로 조절할 수 있게 합니다.                                           │
│  ([출처](https://www.ibm.com/think/topics/langgraph))                                                           │
│                                                                                                                 │
│  2. **다중 에이전트 조정**: Langgraph는 여러 에이전트의 상태 및 상호작용을 관리하여 조건과 입력에 따라          │
│  동적으로 조정되는 복잡한 워크플로를 용이하게 합니다.                                                           │
│  ([출처](https://www.datacamp.com/tutorial/langgraph-tutorial))                                                 │
│                                                                                                                 │
│  3. **인간 중심의 피드백**: 사용자로부터의 피드백 메커니즘을 포함하여 Langgraph는 실제 상호작용에 기반하여      │
│  결과를 개선할 수 있어, 반응성이 높은 AI 애플리케이션을 구축하는 데 도움이 됩니다.                              │
│  ([출처](https://www.ibm.com/think/topics/langgraph))                                                           │
│                                                                                                                 │
│  #### 응용 분야                                                                                                 │
│  - **교육**: Langgraph는 학생의 입력과 학습 스타일에 적응하는 지능형 튜터링 시스템을 만드는 데 활용될 수        │
│  있습니다.                                                                                                      │
│  - **언어학**: 연구자들이 언어 구성 요소 간의 관계를 시각화하는 모델을 구축할 수 있도록 하는 상세한 언어        │
│  데이터 분석을 가능하게 합니다.                                                                                 │
│  ([출처](https://medium.com/@ashutoshsharmaengg/getting-started-with-langgraph-a-beginners-guide-to-building-i  │
│  ntelligent-workflows-67eeee0899d0))                                                                            │
│  - **인공지능**: 이 프레임워크는 언어의 맥락적 이해에 기반하여 챗봇, 의사결정 시스템 및 에이전트 기반 시스템의  │
│  생성 지원합니다. ([출처](https://docs.langchain.com/oss/python/langgraph/overview))                            │
│                                                                                                                 │
│  #### 한계점                                                                                                    │
│  Langgraph는 입력 데이터의 질과 모델링 작업의 특정 설정에 따라 성능이 영향을 받을 수 있습니다. 일부 연구자들은  │
│  훈련 데이터셋에 내재된 편향이 왜곡된 출력을 초래할 수 있음을 지적하여, 입력 데이터의 질 관리의 중요성을        │
│  강조하고 있습니다.                                                                                             │

Output()

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 84ccfef0-38f0-4463-b36c-628c25ab288f                                                                     │
│  Agent: RAG Agent                                                                                               │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



┌───────────────────────────── Execution Traces ──────────────────────────────┐
│                                                                             │
│  🔍 Detailed execution traces are available!                                │
│                                                                             │
│  View insights including:                                                   │
│    • Agent decision-making process                                          │
│    • Task execution flow and timing                                         │
│    • Tool usage details                                                     │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘


Would you like to view your execution traces? [y/N] (20s timeout): 

┌───────────────────────── Tracing Preference Saved ──────────────────────────┐
│                                                                             │
│  Info: Tracing has been disabled.                                           │
│                                                                             │
│  Your preference has been saved. Future Crew/Flow executions will not       │
│  collect traces.                                                            │
│                                                                             │
│  To enable tracing later, do any one of these:                              │
│  • Set tracing=True in your Crew/Flow code                                  │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file              │
│  • Run: crewai traces enable                                                │
│                                                  

In [19]:
result.raw

'### Langgraph란 무엇인가요?\n\nLanggraph는 언어 처리 도구로, 다양한 언어적 데이터를 분석하고 시각화할 수 있는 그래프 및 네트워크 기반의 플랫폼입니다. 이를 통해 사용자는 구문, 의미론, 음성학 등 언어의 다양한 요소 간의 관계를 이해할 수 있습니다. Langgraph는 LangChain 에코시스템의 중요한 부분으로, 복잡한 생성적 AI 에이전트 워크플로를 구축하고 배포하는 데 특화되어 있습니다. \n\n#### 주요 기능 및 방법론\n1. **그래프 기반 아키텍처**: Langgraph는 노드가 다양한 작업이나 구성 요소(예: LLM과 상호작용, 데이터 조작 수행)를 나타내고, 엣지가 이들 간의 관계와 워크플로를 정의하는 그래프 모델을 사용합니다. 이 구조는 복잡한 작업과 모델 및 API 간의 상호작용을 효율적으로 조절할 수 있게 합니다. ([출처](https://www.ibm.com/think/topics/langgraph))\n\n2. **다중 에이전트 조정**: Langgraph는 여러 에이전트의 상태 및 상호작용을 관리하여 조건과 입력에 따라 동적으로 조정되는 복잡한 워크플로를 용이하게 합니다. ([출처](https://www.datacamp.com/tutorial/langgraph-tutorial))\n\n3. **인간 중심의 피드백**: 사용자로부터의 피드백 메커니즘을 포함하여 Langgraph는 실제 상호작용에 기반하여 결과를 개선할 수 있어, 반응성이 높은 AI 애플리케이션을 구축하는 데 도움이 됩니다. ([출처](https://www.ibm.com/think/topics/langgraph))\n\n#### 응용 분야\n- **교육**: Langgraph는 학생의 입력과 학습 스타일에 적응하는 지능형 튜터링 시스템을 만드는 데 활용될 수 있습니다.\n- **언어학**: 연구자들이 언어 구성 요소 간의 관계를 시각화하는 모델을 구축할 수 있도록 하는 상세한 언어 데이터 분석을 가능하게 합니다. ([출처](https://m

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 8afd1c50-3507-4d4a-a02b-acec97437ed8                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: ### Langgraph란 무엇인가요?                                                                      │
│                                                                                                                 │
│  Langgraph는 언어 처리 도구로, 다양한 언어적 데이터를 분석하고 시각화할 수 있는 그래프 및 네트워크 기반의       │
│  플랫폼입니다. 이를 통해 사용자는 구문, 의미론, 음성학 등 언어의 다양한 요소 간의 관계를 이해할 수 있습니다.    │
│  Langgraph는 LangChain 에코시스템의 중요한 부분으로, 복잡한 생성적 AI 에이전트 워크플로를 구축하고 배포하는 데  │
│  특화되어 있습니다.                                                                                             │
│                                                                                                                 │
│  #### 주요 기능 및 방법론                                                                                       │
│  1. **그래프 기반 아키텍처**: Langgraph는 노드가 다양한 작업이나 구성 요소(예: LLM과 상호작용, 데이터 조작      │
│  수행)를 나타내고, 엣지가 이들 간의 관계와 워크플로를 정의하는 그래프 모델을 사용합니다. 이 구조는 복잡한       │
│  작업과 모델 및 API 간의 상호작용을 효율적으로 조절할 수 있게 합니다.                                           │
│  ([출처](https://www.ibm.com/think/topics/langgraph))                                                           │
│                                                                                                                 │
│  2. **다중 에이전트 조정**: Langgraph는 여러 에이전트의 상태 및 상호작용을 관리하여 조건과 입력에 따라          │
│  동적으로 조정되는 복잡한 워크플로를 용이하게 합니다.                                                           │
│  ([출처](https://www.datacamp.com/tutorial/langgraph-tutorial))                                                 │
│                                                                                                                 │
│  3. **인간 중심의 피드백**: 사용자로부터의 피드백 메커니즘을 포함하여 Langgraph는 실제 상호작용에 기반하여      │
│  결과를 개선할 수 있어, 반응성이 높은 AI 애플리케이션을 구축하는 데 도움이 됩니다.                              │
│  ([출처](https://www.ibm.com/think/topics/langgraph))                                                           │
│                                                                                                                 │
│  #### 응용 분야                                                                                                 │
│  - **교육**: Langgraph는 학생의 입력과 학습 스타일에 적응하는 지능형 튜터링 시스템을 만드는 데 활용될 수        │
│  있습니다.                                                                                                      │
│  - **언어학**: 연구자들이 언어 구성 요소 간의 관계를 시각화하는 모델을 구축할 수 있도록 하는 상세한 언어        │
│  데이터 분석을 가능하게 합니다.                                                                                 │
│  ([출처](https://medium.com/@ashutoshsharmaengg/getting-started-with-langgraph-a-beginners-guide-to-building-i  │
│  ntelligent-workflows-67eeee0899d0))                                                                            │
│  - **인공지능**: 이 프레임워크는 언어의 맥락적 이해에 기반하여 챗봇, 의사결정 시스템 및 에이전트 기반 시스템의  │
│  생성 지원합니다. ([출처](https://docs.langchain.com/oss/python/langgraph/overview))                            │
│                                                                                                                 │
│  #### 한계점                                                                                                    │
│  Langgraph는 입력 데이터의 질과 모델링 작업의 특정 설정에 따라 성능이 영향을 받을 수 있습니다. 일부 연구자들은  │
│  훈련 데이터셋에 내재된 편향이 왜곡된 출력을 초래할 수 있음을 지적하여, 입력 데이터의 질 관리의 중요성을

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [20]:
query = '테디노트는 누구인가요?'
video_analyzer = Agent(
    role='Video Analyzer',
    goal = f"""
    Analyze youtube videos about user's query: {query} and Analyze Youtube video transcripts and identify main topics,
    key points, and areas needing further research. This is crucial for answering user's query.
    """,
    backstory='Expert in Youtube video analysis with a keen eye for identifying core themes and knowledge gaps',
    verbose=True,
    max_iter=2,
    llm=llm,
    tools=[youtube_retriever]
)

researcher = Agent(
    role='Web Researcher',
    goal=f"Conduct web searches with query to find additional information on topics identified from the video to answer the user's query: {query}",
    backstory='Skilled internet researcher with a talent for finding reliable and relevant information quickly',
    verbose=True,
    llm=llm,
    max_iter=2,
    tools=[web_search]
)

rag_agent = Agent(
    role='RAG Agent',
    goal=f'''Answer user's query: {query} based on video content analysis and addition research
    if resources are not enough to answer the users' question, then you should command other agents for further research.
    ''',
    backstory='You are a helpful RAG Agent who should refer to the analysis of video analyzer and researcher.',
    llm=llm,
    verbose=True
)

In [ ]:
result = crew.kickoff()

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: aebb3569-82c1-4eaa-9a26-ba3b524abc0d                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Video Analyzer                                                                                          │
│                                                                                                                 │
│  Task: Analyze the video transcript.                                                                            │
│      identify main topics, key points for each topic, and                                                       │
│      list questions that need further research to answer user's 테디노트는 누구인가요?.                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

Output()

╭────────────────────────────────────────────────── Tool Error ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Usage Failed                                                                                              │
│  Name: youtube_retriever                                                                                        │
│  Error: HTTP Error 400: Bad Request                                                                             │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: RAG Agent                                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### 테디노트는 누구인가요?                                                                                     │
│                                                                                                                 │
│  테디노트, 본명 테디 리(Teddy Lee)는 현대 아티스트로서 창의적이고 예술적인 작업으로 주목받고 있습니다. 그는     │
│  디지털 아트와 전통적인 예술 기법을 혼합하여 다양한 관객과 소통하며, 그의 작품은 종종 향수, 어린 시절의 기억,   │
│  엉뚱한 요소들을 주제로 하고 있습니다. 이러한 독특한 서사적 스타일은 그가 사용하는 다양한 매체를 통해           │
│  드러납니다.                                                                                                    │
│                                                                                                                 │
│  1. **예술적 정체성**:                                                                                          │
│     테디노트의 예술적 정체성은 개인적 및 집단적 경험을 반영하는 주제에 대한 깊은 Engagement에서 비롯됩니다.     │
│  그의 작품에는 향수를 느끼게 하고 어린 시절을 떠올리게 하는 이미지를 담아 따뜻함과 엉뚱함을 전달합니다. 그의    │
│  예술 여정을 형성하는 구체적인 영향력은 명확히 문서화되어 있지 않지만, 비슷한 주제를 다루는 많은 아티스트들이   │
│  공유하는 감정적 연결 고리를 시사합니다.                                                                        │
│                                                                                                                 │
│  2. **예술 매체**:                                                                                              │
│     테디노트는 디지털 아트와 전통적인 예술 형식을 모두 포함하는 다양한 매체를 활용합니다. 이러한 융합된         │
│  스타일은 그의 아이디어를 표현하는 데 있어 다재다능한 접근을 가능하게 하며, 이를 통해 다양한 관객층에 접근할    │
│  수 있습니다. 여러 작품에서 확인되는 기술의 혼합은 그의 작품의 미적 매력과 서사적 깊이를 더해줍니다.            │
│                                                                                                                 │
│  3. **커뮤니티 상호작용**:                                                                                      │
│     테디노트의 작품에 대한 커뮤니티 반응은 매우 긍정적이며, 그의 작업에 대한 감정적 연결을 강조하는 지지적인    │
│  피드백이 많습니다. 이러한 긍정적인 반응은 그의 예술적 표현 뿐만 아니라, 피드백과 상호작용이 그의 프로젝트      │
│  발전에 영향을 주는 협업 환경을 보여줍니다.                                                                     │
│                                                                                                                 │
│  4. **향후 프로젝트**:                                                                                          │
│     테디노트는 전통적인 예술 형식을 넘어 새로운 프로젝트를 탐구하고 있다는 암시가 있습니다. 잠재적인 전시회 및  │
│  협업에 대한 추측이 돌고 있으며, 이는 그의 예술에 혁신적인 접근법을 도입할 가능성을 시사합니다. 구체적인        │
│  세부사항은 아직 발표되지 않았지만, 그의 향후 작업을 향한 기대감은 그가 계속해서 새로운 창의적 경로를 통해      │
│  영감을 줄 것임을 암시합니다.                                                                                   │
│                                                                                                                 │
│  **참고 자료**:                                                                                                 │
│  - [Teddy Lee GitHub Profile](https://github.com/teddylee777)                                                   │
│  - 커뮤니티 피드백 및 전시회 관련 댓글 (직접 인용은 아니지만 공공 반응을 나타냄)                                │
│                                                                                                                 │
│  결론적으로, 테디노트는 복잡한 주제를 다양한 매체를 통해 엮어내며 개인적이고 공동체적인 연결을 구축하는 현대    │
│  예술가로 자리잡고 있습니다. 그의 작업은 관객들에게 깊은 감정을 불러일으키며 예술의 새로운 미적 경계를          │
│  탐구하는 데 기여하고 있습니다.                                                                                 │
│                                                                                        

Output()

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: e2d4b786-e7c7-4863-b174-93f46df1cfcd                                                                     │
│  Agent: RAG Agent                                                                                               │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

┌───────────────────────────── Execution Traces ──────────────────────────────┐
│                                                                             │
│  🔍 Detailed execution traces are available!                                │
│                                                                             │
│  View insights including:                                                   │
│    • Agent decision-making process                                          │
│    • Task execution flow and timing                                         │
│    • Tool usage details                                                     │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
Would you like to view your execution traces? [y/N] (20s timeout): 



┌───────────────────────── Tracing Preference Saved ──────────────────────────┐
│                                                                             │
│  Info: Tracing has been disabled.                                           │
│                                                                             │
│  Your preference has been saved. Future Crew/Flow executions will not       │
│  collect traces.                                                            │
│                                                                             │
│  To enable tracing later, do any one of these:                              │
│  • Set tracing=True in your Crew/Flow code                                  │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file              │
│  • Run: crewai traces enable                                                │
│                                                                             │
└─────────────────────────────────────

In [24]:
result.raw

'### 테디노트는 누구인가요?\n\n테디노트, 본명 테디 리(Teddy Lee)는 현대 아티스트로서 창의적이고 예술적인 작업으로 주목받고 있습니다. 그는 디지털 아트와 전통적인 예술 기법을 혼합하여 다양한 관객과 소통하며, 그의 작품은 종종 향수, 어린 시절의 기억, 엉뚱한 요소들을 주제로 하고 있습니다. 이러한 독특한 서사적 스타일은 그가 사용하는 다양한 매체를 통해 드러납니다.\n\n1. **예술적 정체성**:  \n   테디노트의 예술적 정체성은 개인적 및 집단적 경험을 반영하는 주제에 대한 깊은 Engagement에서 비롯됩니다. 그의 작품에는 향수를 느끼게 하고 어린 시절을 떠올리게 하는 이미지를 담아 따뜻함과 엉뚱함을 전달합니다. 그의 예술 여정을 형성하는 구체적인 영향력은 명확히 문서화되어 있지 않지만, 비슷한 주제를 다루는 많은 아티스트들이 공유하는 감정적 연결 고리를 시사합니다.\n\n2. **예술 매체**:  \n   테디노트는 디지털 아트와 전통적인 예술 형식을 모두 포함하는 다양한 매체를 활용합니다. 이러한 융합된 스타일은 그의 아이디어를 표현하는 데 있어 다재다능한 접근을 가능하게 하며, 이를 통해 다양한 관객층에 접근할 수 있습니다. 여러 작품에서 확인되는 기술의 혼합은 그의 작품의 미적 매력과 서사적 깊이를 더해줍니다.\n\n3. **커뮤니티 상호작용**:  \n   테디노트의 작품에 대한 커뮤니티 반응은 매우 긍정적이며, 그의 작업에 대한 감정적 연결을 강조하는 지지적인 피드백이 많습니다. 이러한 긍정적인 반응은 그의 예술적 표현 뿐만 아니라, 피드백과 상호작용이 그의 프로젝트 발전에 영향을 주는 협업 환경을 보여줍니다.\n\n4. **향후 프로젝트**:  \n   테디노트는 전통적인 예술 형식을 넘어 새로운 프로젝트를 탐구하고 있다는 암시가 있습니다. 잠재적인 전시회 및 협업에 대한 추측이 돌고 있으며, 이는 그의 예술에 혁신적인 접근법을 도입할 가능성을 시사합니다. 구체적인 세부사항은 아직 발표되지 않았지만, 그의 향후 작업을

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: aebb3569-82c1-4eaa-9a26-ba3b524abc0d                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: ### 테디노트는 누구인가요?                                                                       │
│                                                                                                                 │
│  테디노트, 본명 테디 리(Teddy Lee)는 현대 아티스트로서 창의적이고 예술적인 작업으로 주목받고 있습니다. 그는     │
│  디지털 아트와 전통적인 예술 기법을 혼합하여 다양한 관객과 소통하며, 그의 작품은 종종 향수, 어린 시절의 기억,   │
│  엉뚱한 요소들을 주제로 하고 있습니다. 이러한 독특한 서사적 스타일은 그가 사용하는 다양한 매체를 통해           │
│  드러납니다.                                                                                                    │
│                                                                                                                 │
│  1. **예술적 정체성**:                                                                                          │
│     테디노트의 예술적 정체성은 개인적 및 집단적 경험을 반영하는 주제에 대한 깊은 Engagement에서 비롯됩니다.     │
│  그의 작품에는 향수를 느끼게 하고 어린 시절을 떠올리게 하는 이미지를 담아 따뜻함과 엉뚱함을 전달합니다. 그의    │
│  예술 여정을 형성하는 구체적인 영향력은 명확히 문서화되어 있지 않지만, 비슷한 주제를 다루는 많은 아티스트들이   │
│  공유하는 감정적 연결 고리를 시사합니다.                                                                        │
│                                                                                                                 │
│  2. **예술 매체**:                                                                                              │
│     테디노트는 디지털 아트와 전통적인 예술 형식을 모두 포함하는 다양한 매체를 활용합니다. 이러한 융합된         │
│  스타일은 그의 아이디어를 표현하는 데 있어 다재다능한 접근을 가능하게 하며, 이를 통해 다양한 관객층에 접근할    │
│  수 있습니다. 여러 작품에서 확인되는 기술의 혼합은 그의 작품의 미적 매력과 서사적 깊이를 더해줍니다.            │
│                                                                                                                 │
│  3. **커뮤니티 상호작용**:                                                                                      │
│     테디노트의 작품에 대한 커뮤니티 반응은 매우 긍정적이며, 그의 작업에 대한 감정적 연결을 강조하는 지지적인    │
│  피드백이 많습니다. 이러한 긍정적인 반응은 그의 예술적 표현 뿐만 아니라, 피드백과 상호작용이 그의 프로젝트      │
│  발전에 영향을 주는 협업 환경을 보여줍니다.                                                                     │
│                                                                                                                 │
│  4. **향후 프로젝트**:                                                                                          │
│     테디노트는 전통적인 예술 형식을 넘어 새로운 프로젝트를 탐구하고 있다는 암시가 있습니다. 잠재적인 전시회 및  │
│  협업에 대한 추측이 돌고 있으며, 이는 그의 예술에 혁신적인 접근법을 도입할 가능성을 시사합니다. 구체적인        │
│  세부사항은 아직 발표되지 않았지만, 그의 향후 작업을 향한 기대감은 그가 계속해서 새로운 창의적 경로를 통해      │
│  영감을 줄 것임을 암시합니다.                                                                                   │
│                                                                                                                 │
│  **참고 자료**:                                                                                                 │
│  - [Teddy Lee GitHub Profile](https://github.com/teddylee777)                                                   │
│  - 커뮤니티 피드백 및 전시회 관련 댓글 (직접 인용은 아니지만 공공 반응을 나타냄)                                │
│                                                                                                                 │
│  결론적으로, 테디노트는 복잡한 주제를 다양한 매체를 통해 엮어내며 개인적이고 공동체적인 연결을 구축하는 현대    │
│  예술가로 자리잡고 있습니다. 그의 작업은 관객들에게 깊은 감정을 불러일으키며 예술의 새로운 미적 경계를          │
│  탐구하는 데 기여하고 있습니다.                                                       

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯